# silver_create_member
Bronze → silver for members. This notebook holds the SQL (`%sql` cells) and the run order.
The Python logic is in three modules in `src/skypoints/silver/`:

| Module | What |
|---|---|
| `clean_and_validate.py` | text → types, reject rules, age / stale, latest record per member |
| `country_tables.py` | country list, split into one table per country (with its MERGE statements), backfill, reconciliation |
| `process_member_batch.py` | what runs for each micro-batch, reject table writer |

Run the cells from top to bottom. The country tables are created by the pipeline when a country needs one.

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import os, sys
sys.path.append(os.path.abspath(".."))

In [0]:
# 2. imports and the stream settings

from pipeline_utils.silver.process_member_batch import load_member_batch_to_silver



In [0]:
BRONZE_TABLE    = "skypoints.bronze.raw_member"
CHECKPOINT_PATH = "/Volumes/skypoints/silver/members/member_checkpoints"

In [0]:
%sql
-- create: silver quarantine: member rows that failed a validation rule. Nothing is dropped silently.
CREATE TABLE IF NOT EXISTS skypoints.silver.member_rejects
(
    member_name         STRING     COMMENT 'Trimmed raw value, can be NULL',
    member_id           STRING     COMMENT 'Trimmed raw value, can be NULL',
    reject_reason       STRING     COMMENT 'One or more codes written as column:CHECK, joined with "; "',
    _rescued_data       STRING     COMMENT 'The full raw bronze row as JSON',
    source_file_name    STRING,
    file_date           DATE,
    ingestion_timestamp TIMESTAMP  COMMENT 'Bronze load time of the rejected row',
    rejected_at         TIMESTAMP  COMMENT 'When silver rejected the row'
)
USING DELTA
COMMENT 'Silver quarantine: member rows that failed a validation rule. Nothing is dropped silently.'

In [0]:
%sql
-- create: current country per member.
CREATE TABLE IF NOT EXISTS skypoints.silver.member_country_lookup
(
    member_name   STRING  COMMENT 'Key. One row per member',
    country_code  STRING  COMMENT 'The country the member currently lives in',
    file_date     DATE    COMMENT 'Feed date of the record that set this value. Stops old files overwriting newer data'
)
USING DELTA
CLUSTER BY (member_name)
COMMENT 'Silver: current country per member '

In [0]:
# 5. process everything new in bronze, then stop
query = (
    spark.readStream.table(BRONZE_TABLE)
    .writeStream
    .foreachBatch(load_member_batch_to_silver)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

In [0]:
%sql
-- check: why rows were rejected (a row with several problems appears once, with all its codes)
SELECT reject_reason, count(*) AS row_count
FROM skypoints.silver.member_rejects
GROUP BY reject_reason
ORDER BY row_count DESC

In [0]:
%sql
select reject_reason , count(*) from skypoints.silver.member_rejects
group by reject_reason;

In [0]:
%sql
select * from skypoints.silver.member_rejects
where reject_reason = 'member_name:MISSING; member_id:MISSING; country_code:MISSING; enrollment_date:MISSING';

In [0]:
%sql
-- check: the latest rejected rows
SELECT count(*) FROM skypoints.silver.member_test;

In [0]:
%sql
-- check: members per country, as member_country_lookup sees them
SELECT *
FROM skypoints.bronze.raw_member
WHERE member_id IS NULL;


In [0]:
%sql
select * from skypoints.silver.member_rejects
where reject_reason = 'enrollment_date:INVALID_DATE; last_flight_date:INVALID_DATE; state:INVALID_FORMAT';